# LABB - Tre i rad

I den här laborationen kommer du att utforska kunskapsbaserade AI-agenter för tvåspelarspel. Mer specifikt hur kunskapsbaserade AI-agenter kan använda resonerande och planering för att spela tre i rad (Tic-Tac-Toe).

Laborationen introducerar först spelet tre i rad (som Python-kod) och går sedan igenom ett antal olika typer av AI-agenter:
* Random Agent - slumpmässig
* Rule-based Agent - regelbaserad
* Planning Agent (Search) - planerande
* Adverserial Planning Agent (Minimax) - adversariell planerande agent
och uppmanar dig att implementera din egen regelbaserade agent.

Som en bonusuppgift utmanas du (inget krav!) att implementera en adversariell stokastisk planerande agent genom att implementera Monte Carlo Tree Search (MCTS).

**Mål:** Undersöka begränsningar, styrkor och svagheter hos olika AI-agentansatser och metoder inom tre i rad-domänen.

**Begreppslista:**

* Agent: Ett program som observera spelet och tar egna beslut.

* Adversariell (Adversarial): Att spela mot en motståndare med motsatta mål (nollsummespel).

* Stokastisk: Slumpmässig / baserad på sannolikheter.

* Tillståndsrymd (State space): Alla möjliga positioner på brädet.

**Regler**

* TILLÅTET: Att använda AI för att generera kod block för block, felsöka Python-kod eller förklara hur algoritmer (som Minimax och MCTS) fungerar teoretiskt.

* FÖRBJUDET: Att låta AI svara på reflektionsfrågorna. Svaren ska baseras på dina egna observationer när du faktiskt spelar mot agenterna.

**Gör labben uppifrån och ned.**
<br />
**Du behöver inte läsa längre fram än till det aktuella labbmomentet.**

<br />
Om du skulle råka redigera koden på sådant sätt att den fastnar i en oändlig loop (aldrig blir klar) måste du klicka på &#10227; (Restart the kernel)
<hr />

### **Authors**
Mathias Ahlgren <br />
Mattias Tiger, mattias.tiger@liu.se

### **License**
CC BY-NC-SA 4.0 <br />
https://creativecommons.org/licenses/by-nc-sa/4.0/

The lab (Notebook: code and instructions) is free to use, share, change and to make your own version suitable for your own students. The only restriction to this version and any new version is that 1) the same license (CC BY-NC-SA 4.0) is applied, 2) that the above mentioned authors are referred to as the authors of the original version and 3) that it cannot be used in a setting where a person or business has to pay to get access to, or in order to use, the version. I.e. full permission is given to use the lab, and subsequent versions, in schools or study circles, even running on paid-for computational resources (e.g. paid version of Google colab).
<hr />

# Tre i rad - Labb 1.0 - Spelet
Vi börjar med att ställa in spelplanen, skapa det grafiska gränssnittet och lägga till händelselyssnare för knapparna.

### Steg 1: Konfiguration och initiering

* **Uppgift 1.0.1:** Gör en snabb genomläsning av koden. Fokusera på funktionsnamn och beskrivningar.

  * Fråga 1: Vad är skillnaden mellan *check_winner(...)* och *check_tile_status(...)*?
  * Fråga 2: Vad avgör spelplanens storlek?

* **Uppgift 1.0.2:** Kör kodcellen (play-ikonen längst upp till vänster) för att initiera spelet.

In [ ]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import random
import sys
import threading
from time import perf_counter, sleep

# Board configuration
BOARD_SIZE = 3
board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)

def check_winner(board):
    """
    Checks if there is a winner on the board

    Args:
        board: Current game board (2D array).

    Returns:
      1 for player1
      2 for player2
      3 for Draw
      None for ongoing.
    """

    # Check rows and columns for a winner
    for i in range(BOARD_SIZE):
        if np.all(board[i, :] == board[i, 0]) and board[i, 0] != 0:
            return board[i, 0]
        if np.all(board[:, i] == board[0, i]) and board[0, i] != 0:
            return board[0, i]

    # Check diagonals for a winner
    if np.all(np.diag(board) == board[0, 0]) and board[0, 0] != 0:
        return board[0, 0]
    if np.all(np.diag(np.fliplr(board)) == board[0, BOARD_SIZE - 1]) and board[0, BOARD_SIZE - 1] != 0:
        return board[0, BOARD_SIZE - 1]

    # Check for a draw (no winner and no empty spaces)
    if not np.any(board == 0):
        return 3

    return None

def check_tile_status(board, row, col, player):
    """
    Simulates placing a move at the specified (row, col) and checks if it results in a win or draw.

    Args:
        board: Current game board (2D array).
        row: Row index of the tile to check.
        col: Column index of the tile to check.
        current_player: The player making the move (1 or 2).

    Returns:
        1 if Player 1 wins,
        2 if Player 2 wins,
        3 for a Draw,
        None for ongoing or if the tile is occupied.
    """

    if board[row, col] != 0:
        return None  # The tile is already filled, no result.

    # Temporarily place the move for current_player
    board[row, col] = player

    # Check row for a winner
    if np.all(board[row, :] == player):
        board[row, col] = 0  # Undo the move
        return player

    # Check column for a winner
    if np.all(board[:, col] == player):
        board[row, col] = 0  # Undo the move
        return player

    # Check main diagonal if applicable
    if row == col and np.all(np.diag(board) == player):
        board[row, col] = 0  # Undo the move
        return player

    # Check anti-diagonal if applicable
    if row + col == BOARD_SIZE - 1 and np.all(np.diag(np.fliplr(board)) == player):
        board[row, col] = 0  # Undo the move
        return player

    # Check for a draw (no winner and no empty spaces)
    if not np.any(board == 0):
        board[row, col] = 0  # Undo the move
        return 3  # Draw

    board[row, col] = 0  # Undo the move if no win/draw
    return None

players = [
        ('Human', 'human')
    ]

# Dropdown menu for player1
player1_dropdown = widgets.Dropdown(
    options=players,
    value='human',
    description='Player1 (X)',
)

# Dropdown menu for player2
player2_dropdown = widgets.Dropdown(
    options=players,
    value='human',
    description='Player2 (0)',
)

# Initialize the global status label here
ai_status_label = widgets.Label(value="")
# Placeholder in the game GUI where the AI progress bar is shown
ai_progress_box = widgets.VBox()

def create_buttons():
    """ Creates and returns a list of buttons """
    return [[widgets.Button(description='', layout=widgets.Layout(width='80px', height='80px')) for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]

buttons = create_buttons()

def on_button_click(i, j):
    """ Event for making the move as a human and playing the Ai's next move """
    global current_player, board

    if board[i, j] == 0:  # If the cell is empty
        board[i, j] = current_player
        buttons[i][j].description = 'X' if current_player == 1 else 'O'
        buttons[i][j].disabled = True
        buttons[i][j].style.button_color = 'black'
        buttons[i][j].style.text_color = 'white'
        buttons[i][j].style.font_weight = 'bold'

        winner = check_winner(board)
        if winner:
            display_result(winner)
            return

        # Switch turns
        current_player = 1 if current_player == 2 else 2

        # If the current player is human we stop to wait for player to click a tile
        if get_current_player_type() == 'human':
            return

        # Stop and wait for the "step" button
        if player1_dropdown.value != 'human' and player2_dropdown.value != 'human':
            return

        # If it's an AI move, make the move immediately
        if get_current_player_type() != 'human':
            ai_move = get_ai_move(get_current_player_dropdown())
            if ai_move:
                on_button_click(ai_move[0], ai_move[1])

def get_ai_move(player_dropdown):
    global ai_status_label, current_player, board

    """
    Gets the AI's next move based on the chosen strategy.
    Displays a progress bar and elapsed time during AI calculation and shows the total time taken.
    """

    ai_move = None
    selected_strategy = player_dropdown.value
    ai_status_label.value = ""

    # Create a progress bar widget
    progress_bar = widgets.IntProgress(
        value=0,
        min=0,
        max=100,
        description='AI Thinking...',
        bar_style='info',
        orientation='horizontal'
    )

    # Show the progress bar in the game GUI
    ai_progress_box.children = [progress_bar]

    # Start the timer
    start_time = perf_counter()

    # Function to perform AI calculation
    def ai_calculation():
        nonlocal ai_move
        # AI decision-making process based on strategy
        if selected_strategy == 'random':
            ai_move = find_best_move_random(board)
        elif selected_strategy == 'my_ai_strategy':
            ai_move = find_best_move(board, current_player)
        elif selected_strategy == 'rule_based':
            ai_move = find_best_move_rule_based(board, current_player)
        elif selected_strategy == 'minimax':
            ai_move = find_best_move_min_max(board, current_player, use_alpha_beta=True)
        elif selected_strategy == 'dfs':
            ai_move = find_best_move_dfs(board, current_player)
        elif selected_strategy == 'bfs':
            ai_move = find_best_move_bfs(board, current_player)

    if sys.platform == 'emscripten':
        # Pyodide (JupyterLite) cannot start threads, so calculate directly
        ai_calculation()
    else:
        # Start AI calculation in a separate thread
        ai_thread = threading.Thread(target=ai_calculation)
        ai_thread.start()

        # Update the progress bar and time label in the main thread
        while ai_thread.is_alive():
            progress_bar.value = (progress_bar.value + 1) % progress_bar.max
            elapsed_time = perf_counter() - start_time
            ai_status_label.value = f"Elapsed time: {elapsed_time:.2f} seconds"
            sleep(0.1)

        # Ensure the AI thread has finished
        ai_thread.join()

    # Close the progress bar and update the time label one last time
    ai_progress_box.children = []
    progress_bar.close()
    elapsed_time = perf_counter() - start_time
    ai_status_label.value = f"Total time taken: {elapsed_time:.2f} seconds"

    # Display the total time taken by AI to make its move
    ai_status_label.value = f"AI took {elapsed_time:.2f} seconds to make its move."

    return ai_move

def get_current_player_type():
    """
    Determines if the current player is AI or human.
    """
    return player1_dropdown.value if current_player == 1 else player2_dropdown.value

def get_current_player_dropdown():
    """
    Gets the appropriate dropdown for the current player (Player 1 or Player 2).
    """
    return player1_dropdown if current_player == 1 else player2_dropdown

def display_result(winner):
    """
    Displays the result of the game.
    """
    result = ""
    if winner == 1:
        result = "Player 1 wins" if player1_dropdown.value == 'human' else "AI (Player 1) wins"
    elif winner == 2:
        result = "Player 2 wins" if player2_dropdown.value == 'human' else "AI (Player 2) wins"
    elif winner == 3:
        result = "It's a draw"

    result_label.value = result
    disable_all_buttons()

def disable_all_buttons():
    """
    Disables all buttons.
    """
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            buttons[i][j].disabled = True

def step_game(_=None):
    global current_player
    """
    Function to step through the AI vs AI moves when the "Step" button is pressed.
    """
    if player1_dropdown.value != 'human' and player2_dropdown.value != 'human': # If Ai vs Ai we diable buttons
        disable_all_buttons()

    if get_current_player_type() != 'human':
        ai_move = get_ai_move(get_current_player_dropdown())
        if ai_move:
            on_button_click(ai_move[0], ai_move[1])

def new_game(_=None, container=None):
    """
    Starts and draws the new game.
    The GUI is drawn in container, a new one is displayed if none is given.
    """
    global board, current_player, buttons, result_label, ai_status_label

    board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)
    current_player = 1

    buttons = create_buttons()

    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            buttons[i][j].on_click(lambda btn, i=i, j=j: on_button_click(i, j))

    grid = widgets.GridBox(children=[button for row in buttons for button in row],
                          layout=widgets.Layout(grid_template_columns=f"repeat({BOARD_SIZE}, 90px)"))

    result_label = widgets.Label(value="") #
    ai_status_label.value = ""

    new_game_button = widgets.Button(description="New game", layout=widgets.Layout(width='200px', height='40px'))
    step_button = widgets.Button(description="Step", layout=widgets.Layout(width='200px', height='40px'))

    if container is None:
        container = widgets.VBox()
        display(container)

    new_game_button.on_click(lambda _: new_game(container=container))
    step_button.on_click(step_game)

    buttons_box = widgets.HBox([new_game_button, step_button], layout=widgets.Layout(justify_content='flex-start'))

    container.children = [player1_dropdown, player2_dropdown, grid, result_label,
                          ai_status_label, ai_progress_box, buttons_box]

### Steg 2: Skapa det grafiska gränssnittet

(Kör kodcellen nedan)

* **Uppgift 1.0.3:** Spela spelet mot dig själv.
  * Fråga 3: Verkar spelet enkelt att vinna?
  * Fråga 4: Om du spelar spelet perfekt som båda spelare, tror du att någon spelare någonsin kommer att vinna spelet?
  * Fråga 5: Tror du att placeringen av det första draget påverkar om ett spel vinns eller förloras, förutsatt perfekta spelare?

In [ ]:
BOARD_SIZE = 3
board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)

player1_dropdown.options = [
      ('Human', 'human')
  ]
player2_dropdown.options = [
      ('Human', 'human')
  ]

# Start the game
new_game()

# Tre i rad - Labb 1.1 - Random Agent - slumpmässig


* **Uppgift 1.1.1:** Läs koden.
  * Fråga 6: Vad gör *random.choice(available_moves)* ?
* **Uppgift 1.1.2:** Kör kodcellerna och spela mot "Random Agent".
  * Fråga 7: Är det enkelt för dig att vinna mot den?
  * Fråga 8: Påverkas hur svårt det är för dig att vinna av om det är du eller den slumpmässiga agenten som får göra det första draget?
* **Uppgift 1.1.3:** Låt "Random Agent" spela mot sig själv.
  * Fråga 9: Verkar spelsättet intelligent?
  * Fråga 10: Hur skulle en mer intelligent agent bete sig?
  * Fråga 11: Vilka enkla förbättringar (konceptuellt) skulle kunna göra agenten bättre?
  
(Om AI-agenten ska göra det första draget, tryck på **step**.)

In [ ]:
def find_best_move_random(board):
    """
    Example strategy of a random strategy.
    Current player is not used (a dummy input).
    """

    # Get the list of all empty positions on the board
    available_moves = [(i, j) for i in range(BOARD_SIZE) for j in range(BOARD_SIZE) if board[i, j] == 0]

    if available_moves:
        return random.choice(available_moves)

In [ ]:
# Board configuration
BOARD_SIZE = 3
board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)

players = [
    ('Human', 'human'),
    ('Random', 'random')
]

# Dropdown menu for player1 and player2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = players[-1][1]

# Start the game
new_game()

### 💡 Instruktion för att bygga din egen agent (Uppgift 1.2.2)

För att din agent ska fungera tillsammans med spelsystemet måste du förstå tre saker:

1. **Spelplanen (`board`):** Representeras som en 2D-lista med storleken 3x3.
   - En tom ruta markeras med ett blanksteg: `' '`
   - Spelare markeras med `'X'` eller `'O'`
   - *Exempel:* `[['X', ' ', 'O'], [' ', 'X', ' '], ['O', ' ', ' ']]`
2. **Tillgängliga drag (`available_moves`):** En lista med tupler över koordinater som är lediga:
   - *Exempel:* `[(0, 1), (1, 0), (1, 2), (2, 1), (2, 2)]`
3. **Returvärde:** Din funktion **måste** returnera en tuple med koordinationerna för ditt drag, t.ex. `(1, 1)`.

  

# Tre i rad - Labb 1.2 - Din egen Agent

**Kan du slå din egen agent?**

* **Uppgift 1.2.1:** Läs koden.
  * Fråga 12: Vad gör *Rule 1* och hur fungerar den?
* **Uppgift 1.2.2:** Utöka koden för att få din egen AI-agent att spela spelet.<br/>
  (Minsta krav: Om spelplanens storlek är 3 och spelplanen är tom, gör den första markeringen i mitten av spelplanen.)
  * Fråga 13: Vilka är några (1–3) bra regler? Exempel på idéer:
  Skapa en funktion som kollar om du kan vinna i nästa drag och skapa en funktion som kollar om motståndaren håller på att vinna och blockera.
  * Fråga 14: Vilka regler implementerade du?
  
  Här får du använda AI för att skriva koden.
  * Dålig prompt: "Skriv en funktion i Python som spelar Tre i rad."
  * Bra prompt: "Jag vill bygga en regelbaserad agent i Python för Tre i rad. Jag har identifierat att agenten först ska kolla om den kan vinna i nästa drag. Kan du hjälpa mig att skriva en Python-funktion som bara kollar efter ett vinnande drag utifrån en 3x3-lista?"
* **Uppgift 1.2.3:** Kör kodcellerna och spela mot din egen agent.
  * Fråga 15: Är det enkelt för dig att vinna mot den?
  * Fråga 16: Påverkas hur svårt det är för dig att vinna av om det är du eller din egen agent som får göra det första draget?
* **Uppgift 1.2.4:** Låt din egen agent spela mot sig själv.
  * Fråga 17: Verkar spelsättet intelligent?
  * Fråga 18: Hur skulle en mer intelligent agent bete sig?
  * Fråga 19: Vilka enkla förbättringar (konceptuellt) skulle kunna göra agenten bättre?

  

In [ ]:
def find_best_move(board, current_player):
    """
    Function for the students to modify

    The goal is to implement rules for how AI will play.
    You have already been given the first and last rules.

    Return a tuple of (i, j) that will be the Ai's move. (0, 0) is top left
    """

    # Rule 1: Win if possible
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i, j] == 0:
                if check_tile_status(board, i, j, current_player) == current_player:
                    return (i, j)  # Winning move found

    # Rule 2: Block opponent's winning move
    opponent = 1 if current_player == 2 else 2  # Identifiera motståndaren
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i, j] == 0:
                if check_tile_status(board, i, j, opponent) == opponent:
                    return (i, j)  # Blockera motståndaren från att vinna

    # Rule 3: Play in the center if possible
    center = BOARD_SIZE // 2  # Index för mitten
    if board[center, center] == 0:
        return (center, center)

 # Rule 4: Play in a corner if possible
    corners = [(0, 0), (0, BOARD_SIZE - 1), (BOARD_SIZE - 1, 0), (BOARD_SIZE - 1, BOARD_SIZE - 1)]
    for corner in corners:
        if board[corner[0], corner[1]] == 0:
            return corner  # Spela i ett hörn om möjligt

    # Last rule any move
    return find_best_move_random(board)

In [ ]:
# Board configuration
BOARD_SIZE = 3
board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)

players = [
    ('Human', 'human'),
    ('Random', 'random'),
    ('My AI Strategy', 'my_ai_strategy')
]

# Dropdown menu for player1 and player2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = players[-1][1]

# Start the game
new_game()

# Tre i rad - Labb 1.3 - Rule-based Agent - regelbaserad

* **Uppgift 1.3.1:** Läs koden.
  * Fråga 20: Förklara den regelbaserade agentens strategi med egna ord.
  * Fråga 21: Vad är skillnaden när det gäller att *thinking ahead* mellan *Rule 1* och *Rule 2*?
  * Fråga 22: Om du måste ta bort två regler, vilka tror du skulle vara minst dåliga att ta bort (försämra spelprestandan minst)?
* **Uppgift 1.3.2:** Kör kodcellerna och spela mot den regelbaserade agenten.
  * Fråga 23: Är det enkelt för dig att vinna mot den?
  * Fråga 24: Påverkas hur svårt det är för dig att vinna av om det är du eller den regelbaserade agenten som får göra det första draget?
* **Uppgift 1.3.3:** Låt den regelbaserade agenten spela mot den slumpmässiga agenten.
  * Fråga 25: Vilken agent verkar mer intelligent?
  * Fråga 26: Spelar det någon roll vilken agent som börjar för hur spelet slutar?
* **Uppgift 1.3.4:** Låt den regelbaserade agenten spela mot sig själv.
  * Fråga 27: Verkar spelsättet intelligent?
  * Fråga 28: Hur skulle en mer intelligent agent bete sig?
  * Fråga 29: Vilka enkla förbättringar (konceptuellt) skulle kunna göra agenten bättre?


In [ ]:
# @title
def find_best_move_rule_based(board, current_player):
    """
    Example function of a rule-based strategy, using check_tile_status.
    """
    opponent = 2 if current_player == 1 else 1

    # Rule 1: Win if possible
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i, j] == 0:
                if check_tile_status(board, i, j, current_player) == current_player:
                    return (i, j)  # Winning move found

    # Rule 2: Block opponent's win
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i, j] == 0:
                if check_tile_status(board, i, j, opponent) == opponent:
                    return (i, j)  # Blocking move found

    # Rule 3: Take the center if available (only for 3x3 board currently)
    if BOARD_SIZE == 3 and board[1, 1] == 0:
        return (1, 1)

    # Rule 4: Take one of the corners if available
    for i, j in [(0, 0), (0, BOARD_SIZE - 1), (BOARD_SIZE - 1, 0), (BOARD_SIZE - 1, BOARD_SIZE - 1)]:
        if board[i, j] == 0:
            return (i, j)

    # Rule 5: Take any available space
    return find_best_move_random(board)

In [ ]:
# @title
# Board configuration
BOARD_SIZE = 3
board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)

players = [
    ('Human', 'human'),
    ('Random', 'random'),
    ('My AI Strategy', 'my_ai_strategy'),
    ('Rule-based', 'rule_based')
]

# Dropdown menu for player1 and player2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = players[-1][1]

# Start the game
new_game()

# Tre i rad - Labb 1.4 - Planning Agent (Search) - planerande

* **Uppgift 1.4.1:** Läs koden.
  * Fråga 30: Förklara de sökbaserade agenternas strategier med egna ord.

* **Uppgift 1.4.2:** Kör kodcellerna och spela mot den sökbaserade agenten.
  * Fråga 31: Är det enkelt för dig att vinna mot dem?
  * Fråga 32: Varför är sökbaserade agenter som bara letar efter vinnande drag inte lämpade för adversariella spel? (spel där två eller flera deltagare tävlar direkt mot varandra och har helt motstridiga mål)
  * Fråga 33: Hur tror du att en sökbaserad strategi skulle kunna förbättras för att spela bättre?



In [ ]:

from collections import deque
import copy

def find_best_move_dfs(board, player, max_depth=6):
    """
    Finds the best move for the AI using Depth-First Search (DFS).
    This function searches the game tree recursively up to a certain depth
    and selects the first move that leads to a win.
    If no winning path is found within the depth limit, it selects a random move.
    """
    opponent = 2 if player == 1 else 1

    def dfs(board, current_player, depth):
        winner = check_winner(board)
        if winner == player:
            return True  # Winning path found
        elif winner == opponent or winner == 3:
            return False  # Opponent wins or draw
        elif depth == max_depth:
            return False  # Max depth reached without finding a win

        for i in range(BOARD_SIZE):
            for j in range(BOARD_SIZE):
                if board[i, j] == 0:
                    # Make a move
                    board[i, j] = current_player
                    found_win = dfs(board, 2 if current_player == 1 else 1, depth + 1)
                    # Undo the move
                    board[i, j] = 0

                    if current_player == player and found_win:
                        return True  # Winning path found
        return False  # No winning path found at this branch

    # Try all possible moves
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i, j] == 0:
                # Make a move
                board[i, j] = player
                found_win = dfs(board, opponent, 1)
                # Undo the move
                board[i, j] = 0

                if found_win:
                    return (i, j)  # Return the first move leading to a win

    # No winning path found, select a random move
    return find_best_move_random(board)

def find_best_move_bfs(board, current_player, max_depth = 5):
    """
    Finds the best move for the AI using Breadth-First Search (BFS).
    This function explores the game tree level by level to find the quickest winning move.
    """
    opponent = 2 if current_player == 1 else 1
    queue = deque()

    # Initialize the queue with all possible moves for the current player
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i, j] == 0:
                new_board = board.copy()
                new_board[i, j] = current_player
                queue.append((new_board, (i, j), current_player, 1))  # (board state, move, player, depth)

    best_move = None
    min_depth = np.inf

    while queue:
        current_board, move, player, depth = queue.popleft()
        winner = check_winner(current_board)

        if winner == current_player:
            # Found a winning move
            if depth < min_depth:
                min_depth = depth
                best_move = move
            continue
        elif winner == opponent or winner == 3:
            continue # skip draws or opponent win

        if depth >= max_depth:
            continue # Stop on max_depth

        # Generate possible moves for the next player
        next_player = opponent if player == current_player else current_player

        for i in range(BOARD_SIZE):
            for j in range(BOARD_SIZE):
                if current_board[i, j] == 0:
                    new_board = current_board.copy()
                    new_board[i, j] = next_player
                    queue.append((new_board, move, next_player, depth + 1))

    if best_move is not None:
        return best_move
    else:
        # If no winning move found, fallback to a random move
        return find_best_move_random(board)



In [ ]:
# Board configuration
BOARD_SIZE = 3
board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)

players = [

    ('Human', 'human'),
    ('Random', 'random'),
    ('My AI Strategy', 'my_ai_strategy'),
    ('Rule-based', 'rule_based'),
    ('Depth first search', 'dfs'),
    ('Breadth first search', 'bfs')
]

# Dropdown menu for player1 and player2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = 'dfs'

# Start the game
new_game()

### 🧠 Hur fungerar Minimax-algoritmen? (Teori inför 1.5)

Minimax beräknar alla möjliga framtida drag genom ett så kallat **beslutsträd**. Algoritmen utgår från att båda spelarna spelar helt optimalt:
* **MAX (AI-agenten):** Vill **maximera** sin slutpoäng (`+1`).
* **MIN (Motståndaren):** Vill **minimera** AI-agentens poäng (`-1`).

#### Beslutsträdet i korthet:
```text
                         [ Vems tur? MAX (AI) ]
                                /     \
                               /       \
               [ Drag A ]     /         \     [ Drag B ]
                             /           \
               [ Vems tur? MIN ]       [ Vems tur? MIN ]
                   /       \               /       \
                 +1         0            -1         +1   <-- Utvärdering vid matchens slut
                  \        /              \        /
            MIN väljer lägsta (0)     MIN väljer lägsta (-1)
                    \                        /
                     \                      /
               MAX väljer högsta poäng: Drag A ger 0, Drag B ger -1.
               ==> AI väljer Drag A!

# Tre i rad - Labb 1.5 - Adverserial Planning Agent (Minimax)




* **Uppgift 1.5.1:** Läs koden.
  * Fråga 34: Vilka är de olika möjliga returvärdena för funktionen *minimax(...)* och vad betyder vart och ett av dem?
  * Fråga 35: Hur många gånger anropas funktionen *minimax(...)* av funktionen *find_best_move_min_max(...)*?
  * Fråga 36: Tror du att *minimax(...)* anropas fler gånger från *minimax(...)* själv än från *find_best_move_min_max(...)*?
  * Fråga 37: Tror du att den här agenten är snabbare, långsammare eller likvärdig när det gäller beräkningstid att köra?
* **Uppgift 1.5.2:** Kör kodcellerna och spela mot den adversariella planerande agenten.
  * Fråga 38: Är det enkelt för dig att vinna mot den?
  * Fråga 39: Påverkas hur svårt det är för dig att vinna av om det är du eller agenten som får göra det första draget?
* **Uppgift 1.5.3:** Låt den adversariella planerande agenten (Minimax) spela mot de andra agenterna.
  * Fråga 40: Vilken agent verkar mer intelligent?
  * Fråga 41: Spelar det någon roll vilken agent som börjar för hur spelet slutar?
  * Fråga 42: Verkar det spela någon roll om du ökar spelplanens storlek till 4x4?
  * Fråga 43: Hur påverkar parametern max_depth minimax-algoritmen?

* **Uppgift 1.5.4:** Låt agenten spela mot sig själv.
  * Fråga 44: Verkar spelsättet intelligent?

In [ ]:
def evaluate_board(board, current_player):
    opponent = 2 if current_player == 1 else 1
    score = 0

    center = (BOARD_SIZE - 1) // 2

    # Positional scores
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i, j] == current_player:
                # Center position
                if i == center and j == center:
                    score += 3  # Assign higher value to center
                # Corner positions
                elif (i == 0 or i == BOARD_SIZE - 1) and (j == 0 or j == BOARD_SIZE - 1):
                    score += 2  # Assign medium value to corners
                else:
                    score += 1  # Assign lower value to edges
            elif board[i, j] == opponent:
                # Subtract positional scores for opponent's pieces
                if i == center and j == center:
                    score -= 3
                elif (i == 0 or i == BOARD_SIZE - 1) and (j == 0 or j == BOARD_SIZE - 1):
                    score -= 2
                else:
                    score -= 1

    # Evaluate potential winning lines for both players
    lines = []

    # Rows and columns
    for i in range(BOARD_SIZE):
        lines.append(board[i, :])    # Row
        lines.append(board[:, i])    # Column

    # Diagonals
    lines.append(board.diagonal())
    lines.append(np.fliplr(board).diagonal())

    for line in lines:
        score += evaluate_line(line, current_player, opponent)

    return score


def evaluate_line(line, player, opponent):
    score = 0
    player_count = np.count_nonzero(line == player)
    opponent_count = np.count_nonzero(line == opponent)
    empty_count = np.count_nonzero(line == 0)

    if player_count == 2 and empty_count == 1:
        score += 10  # Potential winning line for AI
    elif player_count == 1 and empty_count == 2:
        score += 1   # Weak potential line for AI
    elif opponent_count == 2 and empty_count == 1:
        score -= 9   # Potential winning line for opponent (block it)
    elif opponent_count == 1 and empty_count == 2:
        score -= 1   # Weak potential line for opponent

    return score

def minimax(board, depth, is_maximizing, max_depth, current_player, alpha=-np.inf, beta=np.inf, use_alpha_beta=False):
    opponent = 2 if current_player == 1 else 1

    winner = check_winner(board)
    if winner == current_player:
        return 100 - depth  # Prefer quicker wins
    elif winner == opponent:
        return -100 + depth  # Prefer longer losses
    elif winner == 3:
        return 0  # Draw
    elif depth == max_depth:
        # Use the evaluation function
        return evaluate_board(board, current_player)

    if is_maximizing:
        best_score = -np.inf
        for i in range(BOARD_SIZE):
            for j in range(BOARD_SIZE):
                if board[i, j] == 0:
                    board[i, j] = current_player
                    score = minimax(board, depth + 1, False, max_depth, current_player, alpha, beta, use_alpha_beta)
                    board[i, j] = 0
                    best_score = max(score, best_score)
                    if use_alpha_beta:
                        alpha = max(alpha, best_score)
                        if beta <= alpha:
                            break
            if use_alpha_beta and beta <= alpha:
                break
        return best_score
    else:
        best_score = np.inf
        for i in range(BOARD_SIZE):
            for j in range(BOARD_SIZE):
                if board[i, j] == 0:
                    board[i, j] = opponent
                    score = minimax(board, depth + 1, True, max_depth, current_player, alpha, beta, use_alpha_beta)
                    board[i, j] = 0
                    best_score = min(score, best_score)
                    if use_alpha_beta:
                        beta = min(beta, best_score)
                        if beta <= alpha:
                            break
            if use_alpha_beta and beta <= alpha:
                break
        return best_score

def find_best_move_min_max(board, current_player, use_alpha_beta=True, max_depth=8):
    """
    Find the best move using the Minimax algorithm with optional Alpha-Beta pruning.
    The AI evaluates moves by maximizing its own potential and minimizing the opponent's potential.
    """
    best_move = None
    best_score = -np.inf

    if BOARD_SIZE >= 4:
      max_depth = 5

    opponent = 2 if current_player == 1 else 1

    # Initialize alpha and beta values for alpha-beta pruning
    alpha = -np.inf
    beta = np.inf

    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i, j] == 0:
                # Make the move
                board[i, j] = current_player
                # Call minimax recursively and choose the maximum value
                score = minimax(board, 0, False, max_depth, current_player, alpha, beta, use_alpha_beta)
                # Undo the move
                board[i, j] = 0

                if score > best_score:
                    best_score = score
                    best_move = (i, j)

                # Update alpha value if alpha-beta pruning is enabled
                if use_alpha_beta:
                    alpha = max(alpha, best_score)
                    if beta <= alpha:
                        break  # Alpha cutoff
        if use_alpha_beta and beta <= alpha:
            break  # Alpha cutoff

    return best_move




In [ ]:
# Board configuration
BOARD_SIZE = 3
board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)

players = [
    ('Human', 'human'),
    ('Random', 'random'),
    ('My AI Strategy', 'my_ai_strategy'),
    ('Rule-based', 'rule_based'),
    ('Depth first search', 'dfs'),
    ('Breadth first search', 'bfs'),
    ('MinMax', 'minimax'),
]

# Dropdown menu for player1 and player2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = 'minimax'

# Start the game
new_game()

# Tre i rad - Labb 1.6 - Adverserial Stochastic Planning Agent (MCTS)

**(ÖVERKURSEN)**

*Notera att denna notebook körs i Jupyter Lite med en Pyodide kernel och inte CPython, det innebär att den är långsammare så du bör ej jämföra dina resultat med andra om de inte också använt Pyodide. Det går heller inte att använda threads i en Pyodide kernel.*

* **Uppgift 1.6.1:** Implementera MCTS
  * Fråga 45: Vilken källa använde du?
  * Fråga 46: Vilka skillnader jämfört med Minimax förväntar du dig?
* **Uppgift 1.6.2:** Kör kodcellerna och spela mot Adverserial Stochastic Planning Agent (MCTS).
  * Fråga 47: Är det enkelt för dig att vinna mot den?
  * Fråga 48: Påverkas hur svårt det är för dig att vinna av om det är du eller agenten som får göra det första draget?
* **Uppgift 1.6.3:** Låt Adverserial Stochastic Planning Agent (MCTS) spela mot de andra agenterna.
  * Fråga 49: Vilken agent verkar mer intelligent?
  * Fråga 50: Spelar det någon roll vilken agent som börjar för hur spelet slutar?
  * Fråga 51: Spelar det någon roll om du ökar spelplanens storlek till 4x4?
* **Uppgift 1.6.4:** Låt Adverserial Stochastic Planning Agent (MCTS) spela mot sig själv.
  * Fråga 52: Verkar spelsättet intelligent?